In [ ]:
from utils import authenticate, init_vertex_ai  # import the functions to authenticate and initialize Vertex AI
credentials, PROJECT_ID = authenticate()
vertexai = init_vertex_ai(PROJECT_ID, credentials)

/Users/koushikannamalai/Downloads/Github/AI/.venv/lib/python3.13/site-packages/google/cloud/aiplatform/initializer.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources  # noqa: F401 # Note this is used after copybara replacement


In [ ]:
from google.cloud import bigquery
bq_client = bigquery.Client(project=PROJECT_ID, credentials = credentials) # Create a BigQuery client object

In [5]:
QUERY_TABLES = """
SELECT
  table_name
FROM
  `bigquery-public-data.stackoverflow.INFORMATION_SCHEMA.TABLES`
"""
                    

In [7]:
query_job = bq_client.query(QUERY_TABLES)  # Make an API request.
results = query_job.result()  # Wait for the job to complete.

In [ ]:
for row in results:                    # Iterate over the rows of the results
    for value in row.values():
        print(value)

posts_answers
users
posts_orphaned_tag_wiki
posts_tag_wiki
stackoverflow_posts
posts_questions
comments
posts_tag_wiki_excerpt
posts_wiki_placeholder
posts_privilege_wiki
post_history
badges
post_links
tags
votes
posts_moderator_nomination


In [9]:
INSPECT_QUERY = """
SELECT
    *
FROM
    `bigquery-public-data.stackoverflow.posts_questions`
LIMIT 3
"""

In [12]:
import pandas as pd
query_job = bq_client.query(INSPECT_QUERY)


In [13]:
stack_overflow_df = query_job\
    .result()\
    .to_arrow()\
    .to_pandas()
stack_overflow_df.head()

,id,title,body,accepted_answer_id,answer_count,comment_count,community_owned_date,creation_date,favorite_count,last_activity_date,last_edit_date,last_editor_display_name,last_editor_user_id,owner_display_name,owner_user_id,parent_id,post_type_id,score,tags,view_count
0,71443398,Change NgbModal size after open,"<p>I have a modal that initially has a size, b...",71448262.0,1,0,NaT,2022-03-11 18:53:22.020000+00:00,NaN,2022-03-12 08:50:22.153000+00:00,2022-03-11 18:58:48.677000+00:00,None,16466539.0,None,16466539,None,1,0,angular|typescript|bootstrap-modal,256
1,71455993,Returns an undefined value on a post request,<p>When I make a post request returns - undefi...,NaN,1,0,NaT,2022-03-13 10:51:57.207000+00:00,NaN,2022-03-13 13:03:04.647000+00:00,2022-03-13 11:13:25.203000+00:00,None,16800007.0,None,16800007,None,1,0,typescript|deno|oak,256
2,71456969,Pyinstaller gives an error when I install Python,<p>When I try to install I get the following e...,71457345.0,1,0,NaT,2022-03-13 12:57:33.230000+00:00,NaN,2022-03-13 13:43:37.173000+00:00,NaT,None,NaN,None,18453871,None,1,0,python|pyinstaller,256


In [14]:
QUERY_ALL = """
SELECT
    *
FROM
    `bigquery-public-data.stackoverflow.posts_questions` q
"""

In [15]:
query_job = bq_client.query(QUERY_ALL)

In [16]:
try:
    stack_overflow_df = query_job\
    .result()\
    .to_arrow()\
    .to_pandas()
except Exception as e:
    print('The DataFrame is too large to load into memory.', e)

The DataFrame is too large to load into memory. 403 Response too large to return. Consider specifying a destination table in your job configuration. For more details, see https://cloud.google.com/bigquery/troubleshooting-errors

Location: US
Job ID: 905ac3b8-fc0a-47aa-97d8-d23a47513b9c



In [17]:
QUERY = """
SELECT
    CONCAT(q.title, q.body) as input_text,
    a.body AS output_text
FROM
    `bigquery-public-data.stackoverflow.posts_questions` q
JOIN
    `bigquery-public-data.stackoverflow.posts_answers` a
ON
    q.accepted_answer_id = a.id
WHERE
    q.accepted_answer_id IS NOT NULL AND
    REGEXP_CONTAINS(q.tags, "python") AND
    a.creation_date >= "2020-01-01"
LIMIT
    10000
"""

In [18]:
query_job = bq_client.query(QUERY)

In [19]:
### this may take some seconds to run
stack_overflow_df = query_job.result()\
                        .to_arrow()\
                        .to_pandas()

stack_overflow_df.head(2)

,input_text,output_text
0,how to access variables when executing code is...,<p>Although a long time have passed since I as...
1,mognoengine and bson package not work together...,<p>change this </p>\n\n<pre><code>install_requ...


In [20]:
INSTRUCTION_TEMPLATE = f"""\
Please answer the following Stackoverflow question on Python. \
Answer it like you are a developer answering Stackoverflow questions.

Stackoverflow question:
"""

In [21]:
stack_overflow_df['input_text_instruct'] = INSTRUCTION_TEMPLATE + ' '\
    + stack_overflow_df['input_text']

In [22]:
from sklearn.model_selection import train_test_split

In [23]:
train, evaluation = train_test_split(
    stack_overflow_df,
    ### test_size=0.2 means 20% for evaluation
    ### which then makes train set to be of 80%
    test_size=0.2,
    random_state=42
)

In [24]:
import datetime
date = datetime.datetime.now().strftime("%H:%d:%m:%Y")

In [25]:
#Generate a JSON file

cols = ['input_text_instruct', 'output_text']
tune_jsonl  = train[cols].to_json(orient='records', lines=True)

In [26]:
training_data_filename = f"tune_Data_stack_overflow_\
                            python_qa-{date}.jsonl"

In [27]:
with open(training_data_filename, 'w') as f:
    f.write(tune_jsonl)

In [28]:
# Evaluation data set
cols = ['input_text_instruct', 'output_text']
eval_jsonl  = evaluation[cols].to_json(orient='records', lines=True)
eval_data_filename = f"eval_Data_stack_overflow_\
                            python_qa-{date}.jsonl"    
with open(eval_data_filename, 'w') as f:
    f.write(eval_jsonl)